<a href="https://colab.research.google.com/github/Kosuri069/IIT-Patna/blob/main/BookStore_db_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import sqlite3
import pandas as pd

In [28]:
conn = sqlite3.connect("bookstore.db")
cursor = conn.cursor()

print("✓ Database connection established")

✓ Database connection established


In [29]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)
""")

print("✓ Books table created successfully")

✓ Books table created successfully


In [30]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)
""")

print("✓ Customers table created successfully")

✓ Customers table created successfully


In [31]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")

print("✓ Orders table created successfully")

✓ Orders table created successfully


In [32]:
tables = ["books","customers","orders"]

for table in tables:
    print(f"\nSchema for {table} table:")
    schema = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    for row in schema:
        print(row)


Schema for books table:
(0, 'book_id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'author', 'TEXT', 1, None, 0)
(3, 'price', 'REAL', 1, None, 0)
(4, 'stock_quantity', 'INTEGER', 0, '0', 0)

Schema for customers table:
(0, 'customer_id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'email', 'TEXT', 1, None, 0)
(3, 'city', 'TEXT', 0, None, 0)
(4, 'join_date', 'TEXT', 0, None, 0)

Schema for orders table:
(0, 'order_id', 'INTEGER', 0, None, 1)
(1, 'customer_id', 'INTEGER', 0, None, 0)
(2, 'book_id', 'INTEGER', 0, None, 0)
(3, 'quantity', 'INTEGER', 1, None, 0)
(4, 'order_date', 'TEXT', 1, None, 0)
(5, 'total_amount', 'REAL', 0, None, 0)


In [33]:
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]

cursor.executemany("""
INSERT INTO books (title, author, price, stock_quantity)
VALUES (?, ?, ?, ?)
""", books_data)

print("✓ Inserted 5 books")

✓ Inserted 5 books


In [34]:
customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

cursor.executemany("""
INSERT INTO customers (name, email, city, join_date)
VALUES (?, ?, ?, ?)
""", customers_data)

print("✓ Inserted 5 customers")

IntegrityError: UNIQUE constraint failed: customers.email

In [ ]:
orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

cursor.executemany("""
INSERT INTO orders (customer_id, book_id, quantity, order_date, total_amount)
VALUES (?, ?, ?, ?, ?)
""", orders_data)

print("✓ Inserted 7 orders")

In [ ]:
conn.commit()

In [ ]:
print("\nAll Books:")
for row in cursor.execute("SELECT * FROM books"):
    print(row)

print("\nAll Customers:")
for row in cursor.execute("SELECT * FROM customers"):
    print(row)

print("\nAll Orders:")
for row in cursor.execute("SELECT * FROM orders"):
    print(row)

In [ ]:
query = "SELECT * FROM customers WHERE city='Mumbai'"
for row in cursor.execute(query):
    print(row)

In [ ]:
query = """
SELECT * FROM books
WHERE price > 800 AND stock_quantity > 10
"""
for row in cursor.execute(query):
    print(row)

In [ ]:
query = "SELECT COUNT(*) FROM orders"
print("Total Orders:", cursor.execute(query).fetchone()[0])

In [ ]:
query = """
SELECT customer_id, COUNT(*) as order_count
FROM orders
GROUP BY customer_id
ORDER BY order_count DESC
LIMIT 1
"""

print(cursor.execute(query).fetchone())

In [ ]:
query = "SELECT SUM(total_amount) FROM orders"
print("Total Revenue:", cursor.execute(query).fetchone()[0])

In [ ]:
books_df = pd.read_sql("SELECT * FROM books", conn)
customers_df = pd.read_sql("SELECT * FROM customers", conn)
orders_df = pd.read_sql("SELECT * FROM orders", conn)

print(books_df.head(3))
print(customers_df.head(3))
print(orders_df.head(3))

In [ ]:
report = orders_df.merge(customers_df, on="customer_id") \
                  .merge(books_df, on="book_id")

report = report[['order_id','name','city','title','quantity','total_amount']]

report.rename(columns={
    "name":"customer_name",
    "title":"book_title"
}, inplace=True)

report

In [ ]:
avg_order = report['total_amount'].mean()
print("Average Order Value:", avg_order)

In [ ]:
orders_by_city = report.groupby("city")["order_id"].count()
print(orders_by_city)

In [ ]:
popular_book = report.groupby("book_title")["quantity"].sum().sort_values(ascending=False)

print("Most Popular Book:")
print(popular_book.head(1))

In [ ]:
discounts_data = {
    'book_id': [1,2,3,4,5],
    'discount_percent':[10,15,5,20,12]
}

discounts_df = pd.DataFrame(discounts_data)
discounts_df

In [ ]:
discounts_df.to_sql("discounts", conn, if_exists="replace", index=False)

print("✓ Discounts table created")

In [ ]:
pd.read_sql("SELECT * FROM discounts", conn)

In [ ]:
query = """
SELECT
b.title,
b.price AS original_price,
d.discount_percent,
b.price * (1 - d.discount_percent/100.0) AS discounted_price
FROM books b
JOIN discounts d
ON b.book_id = d.book_id
"""

pd.read_sql(query, conn)

In [ ]:
conn.close()
print("✓ Database connection closed")